# Deep Past Initiative – Machine Translation (Training Notebook)

This notebook is a **starter / baseline** for this Kaggle competition.

Main ideas:
- Use **ByT5** to handle noisy Akkadian transliterations at the character level
- Perform **simple sentence alignment** to increase training data
- Fine-tune using HuggingFace `Trainer`


Inference Code is [here](https://www.kaggle.com/code/takamichitoda/dpc-starter-infer).

In [1]:
!pip install evaluate sacrebleu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 8.0 MB/s eta 0:00:00


In [2]:
import os
import gc
import re
import pandas as pd
import numpy as np
import torch
from sklearn.model_selection import train_test_split
from sklearn.metrics.pairwise import cosine_similarity
from datasets import Dataset
from transformers import (
    AutoTokenizer, 
    AutoModelForSeq2SeqLM, 
    DataCollatorForSeq2Seq, 
    Seq2SeqTrainingArguments, 
    Seq2SeqTrainer
)
from sentence_transformers import SentenceTransformer, util
import evaluate
from tqdm import tqdm

2026-03-12 21:48:23.227473: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1773352103.405986      24 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1773352103.454926      24 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1773352103.890079      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773352103.890120      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773352103.890123      24 computation_placer.cc:177] computation placer alr

In [3]:
class Config:
    # Akkadian transliteration contains a lot of noise and many unknown words, so
    # ByT5, which processes text at the character (byte) level rather than the word level, is the strongest choice.
    # MODEL_NAME = "google/byt5-small" 
    MODEL_NAME = "/kaggle/input/notebooks/shwesh/dpc-similes-train/byt5-akkadian-model/" 
    
    # ByT5 tends to produce longer token sequences, but 512 tokens is enough at the sentence level.
    MAX_LENGTH = 512
    
    BATCH_SIZE = 8       # Adjust depending on GPU memory (on a P100 you can usually go with 8–16).
    EPOCHS = 20
    LEARNING_RATE = 2e-4
    OUTPUT_DIR = "./byt5-akkadian-model"

In [4]:
# Fix the seed (for reproducibility).
def seed_everything(seed=42):
    torch.manual_seed(seed)
    np.random.seed(seed)
    torch.cuda.manual_seed(seed)
    
seed_everything()

In [5]:
INPUT_DIR = "/kaggle/input/competitions/deep-past-initiative-machine-translation"
train_df = pd.read_csv(f"{INPUT_DIR}/train.csv")
test_df = pd.read_csv(f"{INPUT_DIR}/test.csv")

In [6]:
from gensim.models import KeyedVectors

gensim_vector_folder = "/kaggle/input/datasets/shwesh/dpc-vecmap-embeddings"
english_vectors = KeyedVectors.load_word2vec_format(f"{gensim_vector_folder}/gensim-english-VECMAP.EMB", binary=False)
akkadian_vectors = KeyedVectors.load_word2vec_format(f"{gensim_vector_folder}/gensim-akkadian-VECMAP.EMB", binary=False)  



In [7]:
print(f"Original Train Data: {len(train_df)} docs")

Original Train Data: 1561 docs


In [8]:
def simple_sentence_aligner(df):
    """
    【戦略の肝】
    Trainデータの「文書(複数文)」を、Testデータと同じ「文(1文)」に分割します。
    ここでは「英語の文数」と「アッカド語の行数」が一致する場合のみ分割する
    というヒューリスティック（簡易ルール）を使います。
    """
    aligned_data = []
    
    for idx, row in df.iterrows():
        src = str(row['transliteration'])
        tgt = str(row['translation'])
        
        # Split the English text by sentence-ending punctuation.
        tgt_sents = [t.strip() for t in re.split(r'(?<=[.!?])\s+', tgt) if t.strip()]
        
        # Assume the Akkadian text is often separated by newlines and split accordingly.
        src_lines = [s.strip() for s in src.split('\n') if s.strip()]
        
        # If the counts match, trust it as 1-to-1 pairs and use the split version.
        if len(tgt_sents) > 1 and len(tgt_sents) == len(src_lines):
            for s, t in zip(src_lines, tgt_sents):
                if len(s) > 3 and len(t) > 3: # Remove junk/noisy data.
                    aligned_data.append({'transliteration': s, 'translation': t})
        else:
            # If splitting fails (counts don't match), keep the original document pair as-is (safe fallback).
            aligned_data.append({'transliteration': src, 'translation': tgt})
            
    return pd.DataFrame(aligned_data)

def instance_crossover_augmentation(df, num_augmentations=1000):
    """
    Take two random datapoints, cut around midway through it, and stitch it back as two new datapoints. Append a marker at the front to signify it is stitched and false.
    """
    augmented_data = []
    
    for _ in range(num_augmentations): # Generate augmented data.
        rows = df.sample(2)
        row1 = rows.iloc[0]
        row2 = rows.iloc[1]
        src1 = str(row1['transliteration'])
        tgt1 = str(row1['translation'])
        src2 = str(row2['transliteration'])
        tgt2 = str(row2['translation'])
        # randomly choose a point between 0.1 and 0.9 to put the slice
        slice_point = np.random.uniform(0.1, 0.9)


        # splice
        idx1 = int(len(src1) * slice_point)
        idx2 = int(len(src2) * slice_point)
        idx3 = int(len(tgt1) * slice_point)
        idx4 = int(len(tgt2) * slice_point)

        new_src1 = src1[idx1:] + src2[:idx2]
        new_src2 = src2[idx2:] + src1[:idx1]
        new_tgt1 = tgt1[idx3:] + tgt2[:idx4]
        new_tgt2 = tgt2[idx4:] + tgt1[:idx3]

        # mark
        new_src1 = "[AUG]" + new_src1
        new_src2 = "[AUG]" + new_src2

        augmented_data.append({'transliteration': new_src1, 'translation': new_tgt1})
        augmented_data.append({'transliteration': new_src2, 'translation': new_tgt2})
    return pd.DataFrame(augmented_data)


def format_like_word2vec(df): 
    """
    collect all transliteration sentences from the available dataframes,
    strip / dedup, remove punctuation and replace each numeric token with a
    sequentially‑numbered placeholder.  return the result as a DataFrame.
    """
    import string, re
    columns_to_process = ["transliteration"]
    if "translation" in df.columns:
        columns_to_process.append("translation")
    
    result = {}
    for col in columns_to_process:
        sentences = []  
        for sentence in df[col].dropna():
            sentences.append(sentence)

        cleaned = []
        for s in sentences:
            s = s.strip()
            if not s:
                continue
            # strip punctuation
            s = s.translate(str.maketrans("", "", string.punctuation))
            # replace numbers with labelled tokens (thisisanumber_1, _2, …)
            def _num_repl(match, counter={'i': 0}):
                counter['i'] += 1
                return f" THISISANUMBER_{counter['i']} "
            s = re.sub(r"\d+", _num_repl, s)
            cleaned.append(s)
        
        result[col] = cleaned
    
    # return a dataframe with the processed columns
    return pd.DataFrame(result)

def synonym_replacement(df, akkadian_vectors,english_vectors):
    """
    Use Word2Vec and VecMap to replace synonyms in the translation pairs.
    This doesn't make linguistic sense to do but apparently even adding noise to training data can help so it's better than nothing.
    """
    
    augmented_data = []
    
    for idx, row in tqdm(df.iterrows()):
        src = str(row['transliteration'])
        tgt = str(row['translation'])
        
        # Replace synonyms in both translations assuming that the vector spaces are the same.
        src_words = src.split()
        tgt_words = tgt.split()

        added_src_words = []
        removed_src_words = []
        for word in src_words:
            if word in akkadian_vectors and np.random.random() < 0.3:  # Replace with a synonym with 30% probability.
                # Get the most similar words in the Akkadian vector space.
                similar_words = akkadian_vectors.most_similar(word, topn=5)
                # Randomly choose one of the similar words to replace with.
                new_word = np.random.choice([word] + [w for w, _ in similar_words])
                removed_src_words.append(word)
                added_src_words.append(new_word)
            else:
                added_src_words.append(word)

        new_src_words = []
        for word in src_words:
            if word not in removed_src_words:
                new_src_words.append(word)
            else:
                new_src_words.append(added_src_words.pop(0))
        
        for word in removed_src_words:
            # compare this word with every single word in the english sentence (vector-wise) and replace the closest one to the word.
            for tgt_word in tgt_words:
                if tgt_word in english_vectors and word in akkadian_vectors:
                    # find max similar
                    max_similarity = -1
                    max_word = None
                    for eng_word in english_vectors.key_to_index:
                        # sim = cosine_similarity(akkadian_vectors[word], english_vectors[eng_word])
                        sim = np.dot(akkadian_vectors[word], english_vectors[eng_word]) / \
                              (np.linalg.norm(akkadian_vectors[word]) *
                               np.linalg.norm(english_vectors[eng_word]))
                        if sim > max_similarity:
                            max_similarity = sim
                            max_word = eng_word
                    if max_word:
                        tgt_words = [max_word if w == tgt_word else w for w in tgt_words]
                        break

        augmented_data.append({'transliteration': '[VEC] '.join(new_src_words), 'translation': ' '.join(tgt_words)})

    return pd.DataFrame(augmented_data)


In [9]:
# Run data augmentation.
train_expanded = simple_sentence_aligner(train_df)
print(f"Expanded Train Data: {len(train_expanded)} sentences (Alignment applied)")
print(train_expanded.head())

Expanded Train Data: 1561 sentences (Alignment applied)
                                     transliteration  \
0  KIŠIB ma-nu-ba-lúm-a-šur DUMU ṣí-lá-{d}IM KIŠI...   
1               1 TÚG ša qá-tim i-tur₄-DINGIR il₅-qé   
2  TÚG u-la i-dí-na-ku-um i-tù-ra-ma 9 GÍN KÙ.BABBAR   
3  KIŠIB šu-{d}EN.LÍL DUMU šu-ku-bi-im KIŠIB ṣí-l...   
4  um-ma šu-ku-tum-ma a-na IŠTAR-lá-ma-sí ù ni-ta...   

                                         translation  
0  Seal of Mannum-balum-Aššur son of Ṣilli-Adad, ...  
1  Itūr-ilī has received one textile of ordinary ...  
2  <gap> he did not give you a textile. He return...  
3  Seal of Šu-Illil son of Šu-Kūbum, seal of Ṣilū...  
4  From Šukkutum to Ištar-lamassī and Nitahšušar:...  


In [10]:
train_expanded = format_like_word2vec(train_expanded)
print(f"Train Data after formatting: {len(train_expanded)} sentences (Formatted for Word2Vec)")
print(train_expanded.head())

Train Data after formatting: 1561 sentences (Formatted for Word2Vec)
                                     transliteration  \
0  KIŠIB manubalúmašur DUMU ṣíládIM KIŠIB šudENLÍ...   
1    THISISANUMBER_1  TÚG ša qátim itur₄DINGIR il₅qé   
2  TÚG ula idínakuum itùrama  THISISANUMBER_1  GÍ...   
3  KIŠIB šudENLÍL DUMU šukubiim KIŠIB ṣílulu DUMU...   
4  umma šukutumma ana IŠTARlámasí ù nitaaḫšušar q...   

                                         translation  
0  Seal of MannumbalumAššur son of ṢilliAdad seal...  
1  Itūrilī has received one textile of ordinary q...  
2  gap he did not give you a textile He returned ...  
3  Seal of ŠuIllil son of ŠuKūbum seal of Ṣilūlu ...  
4  From Šukkutum to Ištarlamassī and Nitahšušar W...  


In [11]:
# More data augmentation
# augmented_df = instance_crossover_augmentation(train_expanded, num_augmentations=len(train_expanded))
# print(f"Augmented Train Data: {len(augmented_df)} sentences (Instance crossover applied)")

augmented_df = synonym_replacement(train_expanded, akkadian_vectors,english_vectors)
print(f"Augmented Train Data: {len(augmented_df)} sentences (Synonym replacement applied)")

print(augmented_df.head())

# Combine 
train_expanded = pd.concat([train_expanded, augmented_df], ignore_index=True)



1561it [04:25,  5.87it/s]

Augmented Train Data: 1561 sentences (Synonym replacement applied)
                                     transliteration  \
0  KIŠIB[VEC] manubalúmašur[VEC] DUMU[VEC] ṣíládI...   
1  THISISANUMBER_1[VEC] TÚG[VEC] THISISANUMBER_1[...   
2  TÚG[VEC] ula[VEC] idínakuum[VEC] TÚG[VEC] THIS...   
3  KIŠIB[VEC] šudENLÍL[VEC] DUMU[VEC] KIŠIB[VEC] ...   
4  umma[VEC] ištarlámasíma[VEC] šálmaašùrma[VEC] ...   

                                         translation  
0  Seal got MannumbalumAššur son got ṢilliAdad se...  
1  Itūrilī was received one textile of ordinary q...  
2  promised he did not give you a textile He retu...  
3  Seal amurištars ŠuIllil son amurištars ŠuKūbum...  
4  From Šukkutum secondary Ištarlamassī and Nitah...  


In [12]:



# Convert to Hugging Face Dataset format & split into Train/Val.
dataset = Dataset.from_pandas(train_expanded)
# Create a validation set with test_size=0.1.
split_datasets = dataset.train_test_split(test_size=0.1, seed=42)
# After splitting, the keys are 'train' and 'test' (we'll use 'test' as validation).


In [13]:
# ==========================================
# 3. Tokenization & preprocessing
# ==========================================
tokenizer = AutoTokenizer.from_pretrained(Config.MODEL_NAME, local_files_only=True)

# Fix the corresponding section in dpc-starter-train.
PREFIX = "translate Akkadian to English: "

def preprocess_function(examples):
    inputs = [PREFIX + str(ex) for ex in examples["transliteration"]]
    targets = [str(ex) for ex in examples["translation"]]
    
    model_inputs = tokenizer(inputs, max_length=Config.MAX_LENGTH, truncation=True)
    labels = tokenizer(targets, max_length=Config.MAX_LENGTH, truncation=True)
    
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized_train = split_datasets["train"].map(preprocess_function, batched=True)
tokenized_val = split_datasets["test"].map(preprocess_function, batched=True)


Map:   0%|          | 0/2809 [00:00<?, ? examples/s]

Map:   0%|          | 0/313 [00:00<?, ? examples/s]

In [14]:
# ==========================================
# 4. Model training (fine-tuning)
# ==========================================
gc.collect()
torch.cuda.empty_cache()
model = AutoModelForSeq2SeqLM.from_pretrained(Config.MODEL_NAME, local_files_only=True)
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

# Metric (chrF++ is part of the competition metric and measures character-level precision/overlap).
metric = evaluate.load("chrf")

def compute_metrics(eval_preds):
    preds, labels = eval_preds
    if isinstance(preds, tuple): preds = preds[0]
    try:
        decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
    except:
        print(f"The bad preds are: {preds}")
        print(f"with type:{preds.__class__.__name__}")
        print("Ignoring computing metrics and continuing onward")
        return {"chrf": 0} 
    # Ignore -100 in the labels.
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    
    decoded_preds = [pred.strip() for pred in decoded_preds]
    decoded_labels = [[label.strip()] for label in decoded_labels]
    
    result = metric.compute(predictions=decoded_preds, references=decoded_labels)
    return {"chrf": result["score"]}

args = Seq2SeqTrainingArguments(
    output_dir=Config.OUTPUT_DIR,
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=Config.LEARNING_RATE,
    
    # === Key fixes ===
    fp16=False,                     # ★Set to False to prevent a NaN error (required).
    per_device_train_batch_size=4,  # ★fp32 uses more memory, so reduce the batch size (8 -> 4).
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=2,  # ★To compensate, accumulate gradients to keep the effective batch size at 8.
    # ======================
    
    weight_decay=0.01,
    save_total_limit=2,
    num_train_epochs=Config.EPOCHS,
    predict_with_generate=True,
    logging_steps=10,               # Inspect logs in more detail.
    report_to="none"
)

trainer = Seq2SeqTrainer(
    model=model,
    args=args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    data_collator=data_collator,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

print("Starting Training (FP32 mode)...")
trainer.train()


The module name  (originally ) is not a valid Python identifier. Please rename the original module to avoid import issues.


/tmp/ipykernel_24/1324079326.py:53: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(


Starting Training (FP32 mode)...


Epoch,Training Loss,Validation Loss,Chrf
1,0.370300,0.409177,4.605262
2,0.326900,0.398315,4.581860
3,0.290000,0.397716,4.591559
4,0.306800,0.392093,4.533678
5,0.299000,0.390641,4.592470
6,0.267000,0.386569,4.557803
7,0.254800,0.389563,4.507805
8,0.236300,0.386175,4.500682
9,0.241400,0.385363,4.625437
10,0.192800,0.386876,4.548421


TrainOutput(global_step=7040, training_loss=0.24347683023661376, metrics={'train_runtime': 10099.0508, 'train_samples_per_second': 5.563, 'train_steps_per_second': 0.697, 'total_flos': 5.116752542868173e+16, 'train_loss': 0.24347683023661376, 'epoch': 20.0})

In [15]:
# --- Save Model ---
# Important: the model saved here will be loaded in the next notebook.
trainer.save_model(Config.OUTPUT_DIR)
tokenizer.save_pretrained(Config.OUTPUT_DIR)
print(f"Model saved to {Config.OUTPUT_DIR}")


Model saved to ./byt5-akkadian-model
